# Automated Commentary with Cortex AI

This notebook walks through the core pattern: take an existing report table,
build a prompt from its columns, call `AI_COMPLETE`, and store the draft for analyst review.

**Prerequisites:** Run `snowflake/build.sh` first to create the demo objects in `LRI_DEMO.LRI_BASE`.

## 1. Setup

In [ ]:
USE WAREHOUSE LRI_DEMO_WH;
USE SCHEMA LRI_DEMO.LRI_BASE;

## 2. Explore the source data

The source table `LIMITS_INDICATORS_REPORT_DATA` holds daily liquidity-risk positions.
Each row is one metric (e.g. Liquidity Coverage Ratio) at one scope (Firm, LOB, or Legal Entity)
for one COB date.

In [ ]:
SELECT METRIC_NAME, DIMENSION_TYPE_CODE, CURRENT_VALUE, REPORTING_UNIT,
       BREACH_LEVEL, BREACH_AMOUNT, HAS_SOURCE_DATA
FROM LIMITS_INDICATORS_REPORT_DATA
WHERE REPORT_AS_OF_COB_DATE = '2026-09-08'
ORDER BY METRIC_KEY
LIMIT 10;

## 3. Look at the prompt

The view `V_PROMPT_ALL` pre-builds a prompt for each row. It concatenates:
- A **system prompt** (preamble + 9 reporting rules)
- A **data dictionary** (DDL column comments)
- The **record values** (labelled key=value pairs)

Let's see what one prompt looks like.

In [ ]:
SELECT METRIC_NAME, STATUS, LENGTH(PROMPT) AS PROMPT_LENGTH, PROMPT
FROM V_PROMPT_ALL
WHERE APPROACH = 'DIRECT_FULL'
  AND REPORT_AS_OF_COB_DATE = '2026-09-08'
  AND METRIC_KEY = 1;

## 4. Single-row AI_COMPLETE

The core pattern is one SQL function call. `AI_COMPLETE` takes a model name and a prompt string,
and returns the model's response. No external API, no SDK, no infrastructure.

In [ ]:
SELECT
  METRIC_NAME,
  AI_COMPLETE('claude-sonnet-5', PROMPT)::VARCHAR AS COMMENTARY
FROM V_PROMPT_ALL
WHERE APPROACH = 'DIRECT_FULL'
  AND REPORT_AS_OF_COB_DATE = '2026-09-08'
  AND METRIC_KEY = 1;

## 5. Batch generation

The stored procedure `SP_GENERATE_STARTER` runs AI_COMPLETE for all rows on a given COB date.
It uses input fingerprinting to skip unchanged rows — if the prompt and model haven't changed,
no model call is made.

In [ ]:
CALL SP_GENERATE_STARTER('2026-09-08', 'claude-sonnet-5');

## 6. Review stored results

Commentary is stored in `AI_COMMENTARY` with the prompt, model, and timestamps.
The `FINAL_COMMENTARY` column is what publishes — it defaults to the AI draft
and can be edited by an analyst.

In [ ]:
SELECT RECORD_UNIQUE_IDENTIFIER, METRIC_NAME, MODEL_NAME,
       GENERATED_AT, IS_EDITED,
       SUBSTR(AI_COMMENTARY, 1, 200) AS COMMENTARY_PREVIEW
FROM AI_COMMENTARY
WHERE APPROACH = 'STARTER'
  AND REPORT_AS_OF_COB_DATE = '2026-09-08'
ORDER BY METRIC_KEY;

## 7. Fingerprint check

Run the procedure again — unchanged rows are skipped. The return message shows how many
model calls were actually made.

In [ ]:
CALL SP_GENERATE_STARTER('2026-09-08', 'claude-sonnet-5');

## 8. Standalone AI_COMPLETE with CONCAT_WS

You don't need the views or stored procedures to use the pattern.
Here's a self-contained SELECT that builds the prompt inline — with the same
system prompt, data dictionary, and key columns used by the demo UI.

In [ ]:
SELECT r.RECORD_UNIQUE_IDENTIFIER,
  AI_COMPLETE('claude-sonnet-5',
    CONCAT_WS(CHR(10),
      -- system prompt: preamble + 9 rules
      'You are a liquidity risk reporting analyst at a global bank, drafting pre-commentary for the daily Limits and Indicators report. Your draft is reviewed and signed off by a human analyst before publication.',
      '',
      'Write commentary for the metric below using ONLY the information provided.',
      '',
      'RULES -- these are not stylistic preferences:',
      '1. Do NOT state, imply, or speculate about WHY the metric moved. No cause is given, so any explanation you supply would be invented. Describe what the position is and how it has changed, never why.',
      '2. Do NOT introduce any figure, date, threshold or percentage that is not stated below.',
      '3. Refer to the event using the exact terminology defined for the metric.',
      '4. Do NOT recommend specific remedial actions, and do NOT name any team, system, desk, client or counterparty.',
      '5. Write in formal third person. No first person, no addressing the reader, and no hedging filler such as "it appears that" or "it would seem".',
      '6. Do NOT use internal system codes, field names or enumerated values in the text (for example BREACH_L2, DETERIORATING, CURRENT_VALUE, NO_SOURCE_DATA). Refer to levels in natural language, such as "a Level 2 breach", "the L2 threshold".',
      '7. Where a figure below is qualified as "at least", you MUST preserve that qualifier.',
      '8. Output plain prose only. No markdown, no bullet points, no headings, no preamble, and no closing summary. Do not restate the metric name as a title.',
      '9. Output ONLY the finished commentary. Never include reasoning, working, self-correction or asides.',
      '',
      -- data dictionary: DDL column comments for selected columns
      'DATA DICTIONARY:',
      'METRIC_NAME: Metric Name: business-facing label for the metric being monitored against a limit/threshold',
      'DIMENSION_TYPE_CODE: Metric Scope Dimension Type Code: organizational dimension the metric is evaluated on. Valid values: FIRM, LOB, LE',
      'LOB_DESCRIPTION: Metric Booking Liquidity LOB Name: name of the line of business for liquidity reporting',
      'LEGAL_ENTITY_DESCRIPTION: Metric Booking Legal Entity Long Name: name of the booking legal entity',
      'CURRENT_VALUE: Current Metric Value: most recent metric value used for evaluation as of the metric COB date',
      'REPORTING_UNIT: Metric Unit Of Measure Code: specific unit used to express the value, e.g. USD, EUR, %, bps',
      'L1_LIMIT_VALUE: Metric Limit Level 1 Value. L1 = early warning / operating tolerance',
      'L2_LIMIT_VALUE: Metric Limit Level 2 Value. L2 = material breach needing management escalation',
      'L3_LIMIT_VALUE: Metric Limit Level 3 Value. L3 = critical breach requiring senior/executive escalation',
      'LIMIT_DIRECTION: Metric Limit Direction Code: + = upper bound (breach when above); - = lower bound (breach when below); +/- = both, breach when outside range',
      'BREACH_LEVEL: Breach Level Code: which threshold level is currently breached (none/L1/L2/L3)',
      'BREACH_AMOUNT: Breach Amount: magnitude of the breach versus the applicable threshold, in the metric unit of measure; null/blank when not breached',
      'CURRENT_UTILIZATION_LEVEL: Current Utilization Level Value: utilization of the current metric value relative to the applicable threshold',
      'HAS_SOURCE_DATA: Has Source Data Indicator: flag indicating whether an upstream sourced value was available for the relevant COB/run',
      '',
      -- record values: key columns
      'RECORD:',
      'METRIC_NAME = ' || COALESCE(TO_VARCHAR(r.METRIC_NAME), '(null)'),
      'DIMENSION_TYPE_CODE = ' || COALESCE(TO_VARCHAR(r.DIMENSION_TYPE_CODE), '(null)'),
      'LOB_DESCRIPTION = ' || COALESCE(TO_VARCHAR(r.LOB_DESCRIPTION), '(null)'),
      'LEGAL_ENTITY_DESCRIPTION = ' || COALESCE(TO_VARCHAR(r.LEGAL_ENTITY_DESCRIPTION), '(null)'),
      'CURRENT_VALUE = ' || COALESCE(TO_VARCHAR(r.CURRENT_VALUE), '(null)'),
      'REPORTING_UNIT = ' || COALESCE(TO_VARCHAR(r.REPORTING_UNIT), '(null)'),
      'L1_LIMIT_VALUE = ' || COALESCE(TO_VARCHAR(r.L1_LIMIT_VALUE), '(null)'),
      'L2_LIMIT_VALUE = ' || COALESCE(TO_VARCHAR(r.L2_LIMIT_VALUE), '(null)'),
      'L3_LIMIT_VALUE = ' || COALESCE(TO_VARCHAR(r.L3_LIMIT_VALUE), '(null)'),
      'LIMIT_DIRECTION = ' || COALESCE(TO_VARCHAR(r.LIMIT_DIRECTION), '(null)'),
      'BREACH_LEVEL = ' || COALESCE(TO_VARCHAR(r.BREACH_LEVEL), '(null)'),
      'BREACH_AMOUNT = ' || COALESCE(TO_VARCHAR(r.BREACH_AMOUNT), '(null)'),
      'CURRENT_UTILIZATION_LEVEL = ' || COALESCE(TO_VARCHAR(r.CURRENT_UTILIZATION_LEVEL), '(null)'),
      'HAS_SOURCE_DATA = ' || COALESCE(TO_VARCHAR(r.HAS_SOURCE_DATA), '(null)')
    )
  )::VARCHAR AS COMMENTARY
FROM LIMITS_INDICATORS_REPORT_DATA r
WHERE r.REPORT_AS_OF_COB_DATE = TO_DATE('2026-09-08')
  AND r.METRIC_KEY = 1;

## 9. Cleanup

When you're done, run `snowflake/cleanup.sql` to drop all demo objects and avoid recurring costs.

```sql
-- Run in a worksheet or via CLI:
-- snow sql -c DEMO -f snowflake/cleanup.sql
```